In [1]:
#-------------------------------------------------------------------------------
# Name:        02_create_input.data
# Purpose:     Create input data for modelling (Slash and Burn)
#
# Author:      Gijs van den Dool & Meggison Oritsemisan
#
# Created:     22/04/2024
# Copyright:   (c) Project Canopy Watch 2024
# Licence:     <Project Canopy Watch>
#-------------------------------------------------------------------------------
# Project Data Folder:
# https://drive.google.com/drive/folders/19wF5gdNnmX4Lwj1kHxltKGRF6DT1yvQ7

# SB buffers:
# This are the inital mask for the model, for which the lines are converted to
# polygons wide enough to capture cells on the perpendicular of the road.
# Suggested distance is: 30m, 3 cells at each side of the road. Wider buffers
# might pick up too much of the stable forests.


In [2]:
import os

import geemap

import json

from shapely.geometry import Point,LineString, Polygon

import numpy as np
import pandas as pd
import geopandas as gpd

from matplotlib import pyplot as plt

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
# Local download of the labels prepared by the team and root of this notebook:
# change the root string to reflect your settings:

project_root = "/content/drive/MyDrive/Project Canopy Official Folder/Task 02 Data Preprocessing/Working Files/Experiment_SB"
task_folder = "01_Input_Data"
file_name = "Iter3_93_SB.geojson" # might be different for other areas

file_path = os.path.join(project_root, task_folder, file_name)

# Test if the path is accessible
if os.path.isfile(file_path):
    print("File found")
else:
    print("File not found")

File found


In [5]:
gdf_SB_lines = gpd.read_file(file_path)

print(len(gdf_SB_lines)) # total lines in the AoI
print(gdf_SB_lines.crs)  # current projection
gdf_SB_lines.head(3)

5
EPSG:4326


,feature_id,image_id_list,image_name,time_frame,min_date,max_date,project_id,sentinel_image_url,geometry
0,clp0w4t7600053j6nhf48izpl,Shifting_cultivation_2_0,93_industrial agriculture_shifting cultivation...,2023-02-04,2023-02-03,2023-02-05,clnfxwt64103z07zmfpm263oy,https://apps.sentinel-hub.com/eo-browser/?zoom...,"POLYGON ((14.81220 -0.02022, 14.81328 -0.02106..."
1,clp0w5eez00063j6n5rjyqtb7,Shifting_cultivation_2_1,93_industrial agriculture_shifting cultivation...,2023-02-04,2023-02-03,2023-02-05,clnfxwt64103z07zmfpm263oy,https://apps.sentinel-hub.com/eo-browser/?zoom...,"POLYGON ((14.82301 -0.03273, 14.82421 -0.03213..."
2,clp0w6bs400083j6n6ll5ehbn,Shifting_cultivation_2_2,93_industrial agriculture_shifting cultivation...,2023-02-04,2023-02-03,2023-02-05,clnfxwt64103z07zmfpm263oy,https://apps.sentinel-hub.com/eo-browser/?zoom...,"POLYGON ((14.84184 -0.03270, 14.84328 -0.03175..."


In [ ]:
def set_projection(gdf_input, org_CRS, new_CRS):
  # working on the projection system (CRS)
  if org_CRS != new_CRS:  # setting the new projection, buffers will be
                              # reprojected to the org_CRS
    gdf_output = gdf_input.to_crs(new_CRS)
  else:
    gdf_output = gdf_input

  print(gdf_output.crs)

  return gdf_output

In [ ]:
def create_buffer(road_line, buffer_distance, AoI_id):
#-------------------------------------------------------------------------------
# Name:        create_buffer
# Purpose:     Create a buffer polygon around all the roads in the label AoI
# Returns:     GeoPandas Dataframe
#-------------------------------------------------------------------------------
  #creating the buffer:
  road_buffer = road_line['geometry'].buffer(buffer_distance)
  road_buffer = gpd.GeoDataFrame(geometry=gpd.GeoSeries(road_buffer))
  road_buffer = road_buffer.rename(columns={0:'geometry'}).set_geometry('geometry')

  #polygon only:
  road_buffer  = road_buffer['geometry'].unary_union

  # converting the polygon to a geodataframe:
  road_buffer = gpd.GeoDataFrame(geometry=gpd.GeoSeries(road_buffer))
  road_buffer = road_buffer.rename(columns={0:'geometry'}).set_geometry('geometry')

  road_buffer["AoI_id"] = AoI_id

  return road_buffer

In [ ]:
buffer_distance = 1 # in meters (ISL = 30)
AoI_id = "SB_93"   # will be different for other areas

# setting the projection
org_CRS = gdf_SB_lines.crs
new_CRS = "EPSG:32632"       # this PROJCRS "WGS 84 / UTM zone 32N" in meters
gdf_SB_lines = set_projection(gdf_SB_lines, org_CRS, new_CRS)

# create the buffer:
gdf_SB_buffer = create_buffer(gdf_SB_lines, buffer_distance, AoI_id)

print(len(gdf_SB_buffer))
gdf_SB_buffer

EPSG:32632
1


,geometry,AoI_id
0,"MULTIPOLYGON (((1151186.379 -3632.934, 1151186...",SB_93


In [ ]:
# reset the projection system and save the buffer:
gdf_SB_buffer = gdf_SB_buffer.set_crs(new_CRS)
gdf_SB_buffer = gdf_SB_buffer.to_crs(org_CRS)

# save the buffer
task_folder = "02_WorkingFiles"
working_folder = os.path.join(project_root, task_folder)
file_name = f"buffer_{AoI_id}.geojson" # might be different for other areas

file_path = os.path.join(working_folder, file_name)
gdf_SB_buffer.to_file(file_path)


In [ ]:
# uncomment to check
#gdf_SB_buffer.explore()

In [ ]:
#-------------------------------------------------------------------------------
# With the buffers created we can go to the next step: creating a grid based
# on the AoI

In [ ]:
def create_grid(AoI_polygon, grid_distance, AoI_id):
#-------------------------------------------------------------------------------
# Name:        create_grid
# Purpose:     Create a grid of polygons inside the AoI based on a grid distance,
#              the grid distance is based on the patch size, and preset to be
#              128x128px, or 1280x1280m
# Returns:     GeoPandas Dataframe
#-------------------------------------------------------------------------------
  minx, miny, maxx, maxy = AoI_polygon.total_bounds

  width = grid_distance
  height = grid_distance
  rows = int(np.ceil((maxy - miny) /  height))
  cols = int(np.ceil((maxx - minx) / width))
  leftOriginX = int(minx)
  rightOriginX = int(minx) + width
  topOriginY = int(maxy)
  bottomOriginY = int(maxy)- height

  polygons = []
  grid_id_list = []
  for i in range(cols):
    topY = topOriginY
    bottomY =bottomOriginY
    for j in range(rows):
      polygons.append(Polygon([(leftOriginX, topY), (rightOriginX, topY),
                              (rightOriginX, bottomY), (leftOriginX, bottomY)]))
      grid_id = f"GID_{AoI_id}_{i}_{j}"
      grid_id_list.append(grid_id)
      topY = topY - height
      bottomY = bottomY - height

    # end for

    leftOriginX = leftOriginX + width
    rightOriginX = rightOriginX + width
  # end for

  grid_polygon = gpd.GeoDataFrame({'geometry':polygons})
  grid_polygon["GID"]=grid_id_list


  return grid_polygon

In [ ]:
working_folder = "01_Input_Data"
file_name = "SB_93.geojson" # might be different for other areas

# get grid file path
grid_path = os.path.join(project_root, working_folder, file_name)

# Test if the path is accessible
os.path.isfile(grid_path)

True

In [ ]:
gdf_AoI_id = gpd.read_file(grid_path)

print(len(gdf_AoI_id)) # total lines in the AoI
print(gdf_AoI_id.crs)  # current projection
gdf_AoI_id.head()
#gdf_AoI_id.explore()

1
EPSG:4326


,image_AoI,min_date,max_date,url,image_date,geometry
0,SB_93,2023-02-03,2023-02-05,https://apps.sentinel-hub.com/eo-browser/?zoom...,2023-02-04,"POLYGON ((14.80807 -0.03900, 14.86543 -0.03900..."


In [ ]:
# setting the projection
#org_CRS = gdf_AoI_id.crs
new_CRS = "EPSG:32632"       # this PROJCRS "WGS 84 / UTM zone 32N" in meters
gdf_AoI_id = set_projection(gdf_AoI_id, org_CRS, new_CRS)


EPSG:32632


In [ ]:
# create the grid:
grid_distance = 5120 # 128px size at 10m ground resolution
gdf_grid = create_grid(gdf_AoI_id, grid_distance, AoI_id)

# reset the projection:
new_CRS = "EPSG:32632"
gdf_grid = gdf_grid.set_crs(new_CRS)
gdf_grid.to_crs(org_CRS, inplace=True)

task_folder = "02_WorkingFiles"
working_folder = os.path.join(project_root, task_folder)
file_name = f"grid_{AoI_id}.geojson" # might be different for other areas

file_path = os.path.join(working_folder, file_name)
gdf_grid.to_file(file_path)

In [ ]:
# uncomment to check
#gdf_grid.explore()

In [ ]:
print(gdf_grid.crs)

#gdf_SB_buffer
print(gdf_SB_buffer.crs)

gdf_mask_full = gpd.overlay(gdf_SB_buffer, gdf_grid, how='union')
gdf_mask_full.head()

EPSG:4326
EPSG:4326


,AoI_id,GID,geometry
0,SB_93,GID_SB_93_0_0,"MULTIPOLYGON (((14.85157 -0.02876, 14.85157 -0..."
1,SB_93,GID_SB_93_1_0,"POLYGON ((14.85386 -0.02801, 14.85383 -0.02800..."
2,NaN,GID_SB_93_0_0,"POLYGON ((14.85383 0.00723, 14.85383 0.00157, ..."
3,NaN,GID_SB_93_0_1,"POLYGON ((14.80806 -0.03886, 14.85383 -0.03885..."
4,NaN,GID_SB_93_1_0,"MULTIPOLYGON (((14.89960 0.00723, 14.89960 -0...."


In [ ]:
# uncomment to check
#gdf_mask_full.explore()

In [ ]:
# In gdf_mask_full both AoI_id	GID	are filled when there is a road buffer in the
# grid, when AoI_id is Null it means that there is not road buffer.
# To select the full GID with a road we first select all the cells which are not
# Null on "AoI_id", and then create a join on GID, selecting also the parts in the
# grid polygon which doesn't have a road buffer.

# xtrain = df.loc[df['Survive'].notnull(), ['Age','Fare', 'Group_Size','deck', 'Pclass', 'Title' ]]
# xtrain

# Select the road elements
xSelect = gdf_mask_full.loc[gdf_mask_full['AoI_id'].notnull(),["AoI_id","GID"]]
xSelect.head(5)



,AoI_id,GID
0,SB_93,GID_SB_93_0_0
1,SB_93,GID_SB_93_1_0


In [ ]:
# The result of the merge with inner is the first step in creating the vector mask
gdf_mask_select= gdf_mask_full.merge(xSelect, on='GID', how='inner', suffixes=('_1', '_2'))

gdf_mask_select[['mask']] = gdf_mask_select[['AoI_id_1']] \
  .where(gdf_mask_select[['AoI_id_1']].isnull(), 1) \
  .fillna(0).astype(int)
gdf_mask_select.drop(['AoI_id_1'], axis=1, inplace=True)

gdf_mask_select = gdf_mask_select.loc[gdf_mask_select['GID'].notnull()]
print(len(gdf_mask_select))
gdf_mask_select.tail(10)


4


/var/folders/hf/1s_l6dt91218_yqmlwxj9pxm0000gn/T/ipykernel_58657/3110764797.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0).astype(int)


,GID,geometry,AoI_id_2,mask
0,GID_SB_93_0_0,"MULTIPOLYGON (((14.85157 -0.02876, 14.85157 -0...",SB_93,1
1,GID_SB_93_1_0,"POLYGON ((14.85386 -0.02801, 14.85383 -0.02800...",SB_93,1
2,GID_SB_93_0_0,"POLYGON ((14.85383 0.00723, 14.85383 0.00157, ...",SB_93,0
3,GID_SB_93_1_0,"MULTIPOLYGON (((14.89960 0.00723, 14.89960 -0....",SB_93,0


In [ ]:
woring_folder = "02_WorkingFiles/"
file_name = f"mask_{AoI_id}.geojson" # might be different for other areas


file_path = os.path.join(project_root, working_folder, file_name)
gdf_mask_select.to_file(file_path)


In [ ]:
# The last step is to create the ImagePatch_AoI, and this is the unmodified
# grid polygon, which is the the GID where mask = 1

# selecting rows based on condition
rslt_df = gdf_mask_select.loc[gdf_mask_select['mask'] == 1, ["GID"]]
rslt_df.head(5)

,GID
0,GID_SB_93_0_0
1,GID_SB_93_1_0


In [ ]:
# The result of the merge with inner is the first step in creating the vector mask
gdf_grid_select= gdf_grid.merge(rslt_df, on='GID', how='inner', suffixes=('_1', '_2'))
gdf_grid_select.head(5)

,geometry,GID
0,"POLYGON ((14.80806 0.00723, 14.85383 0.00723, ...",GID_SB_93_0_0
1,"POLYGON ((14.85383 0.00723, 14.89960 0.00723, ...",GID_SB_93_1_0


In [ ]:
woring_folder = "02_WorkingFiles/"
file_name = f"ImagePatch_{AoI_id}.geojson" # might be different for other areas

# export image patch
file_path = os.path.join(project_root, working_folder, file_name)
gdf_grid_select.to_file(file_path)